# PondSight — train fish detection on a Colab GPU

1. Sign in and choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Run the cells in order. Supply your Roboflow API key in the hidden prompt; it is not saved in this notebook.
3. Download the results before disconnecting. Optional Google Drive backup below preserves checkpoints.

This starts a fresh GPU run; the stopped CPU job had not completed its first epoch.
Training detects **fish**, not species, weight or feeding. No accuracy results have been generated in this notebook yet.


## 1. Install and check GPU

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "roboflow>=1.1,<2" pyyaml
import torch
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU, then rerun. CPU training is disabled."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Download labelled data
[Underwater fish v6](https://universe.roboflow.com/underwater-fish/underwater-fish-detection-izi1l/dataset/6), CC BY 4.0, plus [pond species v1](https://universe.roboflow.com/travaux-professionnel/fish-detection-taxj6-tnm2l/dataset/1), Public Domain. The small supplementary set includes fish outside water. Neither proves accuracy on Nigerian ponds.

In [ ]:
from pathlib import Path
import contextlib, io, getpass, hashlib, json, shutil, uuid
import yaml
from roboflow import Roboflow
ROOT = Path('/content/pondsight')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCES = ['underwater-fish-v6', 'pond-species-v1']
specs = [('underwater-fish', 'underwater-fish-detection-izi1l', 6),
         ('travaux-professionnel', 'fish-detection-taxj6-tnm2l', 1)]
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    api_key = getpass.getpass('Roboflow API key (hidden): ')
if not api_key:
    raise ValueError('A Roboflow API key is required to download the datasets.')
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        rf = Roboflow(api_key=api_key)
        for source, (workspace, project, version) in zip(SOURCES, specs):
            folder = ROOT / 'data' / source
            rf.workspace(workspace).project(project).version(version).download('yolov8', location=str(folder))
except Exception:
    raise RuntimeError('Dataset download failed. Check the API key and network, then rerun this cell.') from None
finally:
    del api_key
print('Both datasets downloaded.')


## 3. Prepare and audit
Species labels become one fish class. Preserve source splits and remove identical files across splits. Similar video frames and augmentations may still cause optimistic scores; separate pond footage is essential.

In [ ]:
def prepare():
    dest = ROOT / 'data' / ('fish-combined-' + uuid.uuid4().hex[:8])
    dest.mkdir(exist_ok=True)
    seen = set()
    counts = {}
    # Held-out images take precedence if a source duplicated an image across splits.
    for split in ('test', 'valid', 'train'):
        images = dest / split / 'images'
        labels = dest / split / 'labels'
        images.mkdir(parents=True, exist_ok=True)
        labels.mkdir(parents=True, exist_ok=True)
        count = 0
        for source in SOURCES:
            base = ROOT / 'data' / source
            metadata = yaml.safe_load((base / 'data.yaml').read_text())
            names = metadata['names']
            for image in sorted((base / split / 'images').glob('*')):
                if image.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
                label = base / split / 'labels' / (image.stem + '.txt')
                if not label.exists():
                    raise ValueError(f'Missing annotation: {label.name}')
                digest = hashlib.sha256(image.read_bytes()).hexdigest()
                if digest in seen: continue
                seen.add(digest)
                rows = []
                for line in label.read_text().splitlines():
                    values = line.split()
                    if len(values) != 5: raise ValueError('Expected detection boxes')
                    cid = int(values[0])
                    if cid < 0 or cid >= len(names): raise ValueError('Invalid class ID')
                    coords = [float(v) for v in values[1:]]
                    if not all(0 <= v <= 1 for v in coords) or min(coords[2:]) <= 0:
                        raise ValueError('Invalid bounding box')
                    rows.append('0 ' + ' '.join(values[1:]))
                filename = source + '_' + image.name
                shutil.copy2(image, images / filename)
                (labels / (Path(filename).stem + '.txt')).write_text('\n'.join(rows))
                count += 1
        counts[split] = count
        if not count: raise ValueError(f'Empty {split} split')
    config = {'path': str(dest), 'train': 'train/images', 'val': 'valid/images',
              'test': 'test/images', 'names': {0: 'fish'}, 'nc': 1}
    path = dest / 'data.yaml'
    path.write_text(yaml.safe_dump(config))
    (dest / 'audit.json').write_text(json.dumps({'counts': counts, 'sources': SOURCES,
        'classes': 'All source species merged to fish; no species identification',
        'deduplication': 'Exact file hashes only; near-duplicate frames may remain'}, indent=2))
    print('Dataset prepared:', counts, flush=True)
    return path


data_yaml = prepare()
print((data_yaml.parent / "audit.json").read_text())

## 4. Checkpoint storage
Set `SAVE_TO_DRIVE = True` to keep checkpoints if Colab disconnects; approve the Google Drive mount yourself. Otherwise download results before the runtime is deleted.

In [ ]:
SAVE_TO_DRIVE = False
from datetime import datetime, timezone
run_name = 'fish_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
project_dir = ROOT / 'runs'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path('/content/drive/MyDrive/PondSight/training')
project_dir.mkdir(parents=True, exist_ok=True)
print('Checkpoints:', project_dir / run_name)


## 5. Train on GPU
50 epochs maximum, early stopping after 10 epochs without improvement. Lower `BATCH` to 4 if GPU memory runs out. This is full fine-tuning, without the CPU run’s frozen backbone.

In [ ]:
from ultralytics import YOLO
EPOCHS, IMAGE_SIZE, BATCH = 50, 640, 8
model = YOLO('yolo11n.pt')
model.train(data=str(data_yaml), epochs=EPOCHS, patience=10,
            imgsz=IMAGE_SIZE, batch=BATCH, device=0, workers=2,
            freeze=0, cache=False, seed=42, deterministic=True,
            project=str(project_dir), name=run_name, save=True,
            save_period=5, plots=True)
run_dir = Path(model.trainer.save_dir)
best = Path(model.trainer.best)
assert best.is_file(), 'Training did not produce best.pt'
print('Best model:', best)


## 6. Evaluate once on the test split
These scores describe these datasets, not your pond. Do not tune settings against the test split. Review separate pond and non-fish videos before using this model operationally.

In [ ]:
trained = YOLO(str(best))
assert set(trained.names.values()) == {'fish'}
metrics = trained.val(data=str(data_yaml), split='test', imgsz=IMAGE_SIZE,
                      batch=BATCH, device=0, workers=2, plots=True)
summary = {'test_mAP50': float(metrics.box.map50),
           'test_mAP50_95': float(metrics.box.map),
           'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
           'note': 'Not validated on Nigerian pond footage. No weight/feeding labels.'}
(run_dir / 'evaluation.json').write_text(json.dumps(summary, indent=2))
shutil.copy2(data_yaml.parent / 'audit.json', run_dir / 'dataset-audit.json')
import importlib.metadata
(run_dir / 'package-versions.json').write_text(json.dumps({name: importlib.metadata.version(name)
    for name in ['ultralytics', 'torch', 'roboflow', 'numpy']}, indent=2))
print(json.dumps(summary, indent=2))


## 7. Download model and evaluation

In [ ]:
import zipfile
from google.colab import files
bundle = ROOT / (run_name + '_results.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(best, 'best.pt')
    for name in ['evaluation.json', 'dataset-audit.json', 'package-versions.json', 'results.csv', 'results.png', 'args.yaml']:
        p = run_dir / name
        if p.is_file(): archive.write(p, name)
files.download(str(bundle))


## Next step: test in PondSight
Extract the downloaded ZIP. After reviewing its evaluation, place `best.pt` at `runs/detect/train/weights/best.pt` in the PondSight project (back up any existing file). Keep `model.fish_classes: ["fish"]`, restart the app, and test both pond and non-fish videos. Bring the ZIP back to this task for review and installation.

**Interrupted run:** if you enabled Drive backup, keep `weights/last.pt`. Reconnect a GPU, reinstall dependencies and recreate data first; use `YOLO(str(checkpoint)).train(resume=True, data=str(data_yaml))` for an unfinished checkpoint, with the same package versions. Do not resume the old CPU run with these different settings.

[Ultralytics training documentation](https://docs.ultralytics.com/modes/train/) · [Colab limits](https://research.google.com/colaboratory/faq.html)
